# Task 1 -- Beam Search vs Greedy Decoding (zero-shot)

Small, throwaway experiment notebook (branch `beamSearchTry`): runs the **zero-shot** Task 1 baseline on a **small subsample** of the test set, comparing three decoding settings:

1. **Greedy** (`Task_1.ipynb`'s current default) -- deterministic, picks the single best next token every step, can't recover from an early mistake.
2. **Beam search** (`num_beams=4`) -- keeps 4 candidate sequences at each step, should find higher joint-probability outputs than greedy.
3. **Beam search + `no_repeat_ngram_size`** -- beam search alone doesn't reliably stop the model's `ppppppp...` repetition loops (all beams tend to converge on the same loop); this constraint forbids repeating a 3-token n-gram, to check whether it's the missing piece.

Only the **zero-shot vanilla model** is evaluated here -- no fine-tuning, no dataset writes, no Hub pushes. Point is to see whether decoding strategy alone moves the needle before spending GPU time changing it in the real training notebooks.

Downloading Libraries and Imports

In [9]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade "pillow<11.0.0" torch torchvision transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv


!pip uninstall -y torchaudio -q


!pip install -q flash-linear-attention

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from tqdm import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor


W0922 09:43:33.772000 12286 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0922 09:43:33.818000 12286 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Setting up environment

In [10]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "beamSearchTry",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    if repo_root.exists():
        # /content survives "Restart session", so an old clone can linger -- pull so
        # the notebook always runs the latest code (restart the session afterwards if
        # eval.utilities was already imported in this kernel).
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent

print(f"Setup Complete. REPO_ROOT: {repo_root}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

from eval.utilities import evaluate_chessboard_model_task_1

print("All custom modules imported successfully!")


Repository directory already exists at: /content/BigDataAndTextMiningProject; pulling latest changes...
Already up to date.
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA L4
All custom modules imported successfully!
Repository directory already exists at: /content/BigDataAndTextMiningProject; pulling latest changes...
Already up to date.
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA L4
All custom modules imported successfully!


Authenticate and load a small subsample of the Task 1 test set

In [11]:
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment -- falling back to interactive login.")
    login()

# Small subsample -- this notebook is about comparing decoding strategies, not
# producing a publishable benchmark number, so a fraction of the full test set
# (400 samples) is enough and keeps each experiment to a couple of minutes.
NUM_SAMPLES = 40

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}' (test split only)...")
test_split_full = load_dataset(dataset_name, split="test", num_proc=16)
test_split = test_split_full.select(range(min(NUM_SAMPLES, len(test_split_full))))

print(f"\nUsing {len(test_split)} / {len(test_split_full)} test samples.")
print(test_split[0])


Logged in to Hugging Face Hub using HF_TOKEN.


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

metadata.jsonl:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

...


Using 40 / 400 test samples.
{'sample_id': 'sample_000000', 'puzzle_id': 'yaQEM', 'task': 'task1', 'fen': '6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of all pieces on the board.', 'target': '6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512 at 0x7F1A77F0A900>, 'file_name_t1': ''}
Logged in to Hugging Face Hub using HF_TOKEN.


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]


Using 40 / 400 test samples.
{'sample_id': 'sample_000000', 'puzzle_id': 'yaQEM', 'task': 'task1', 'fen': '6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of all pieces on the board.', 'target': '6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512 at 0x7F1A76DBEAD0>, 'file_name_t1': ''}


Load the vanilla model

In [4]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("Model and processor loaded correctly!")


Loading model Qwen/Qwen3.5-0.8B...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Model and processor loaded correctly!
Loading model Qwen/Qwen3.5-0.8B...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

## Experiment 1: Greedy decoding (baseline)

Reproduces `Task_1.ipynb`'s current zero-shot behaviour: `generate_kwargs={}` means `evaluate_chessboard_model_task_1` falls back to its default (`max_new_tokens=100`, everything else at the library default of `do_sample=False`, i.e. greedy).

In [5]:
greedy_results_df, greedy_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=test_split,
    model_name="Zero-Shot Greedy",
)

all_results = greedy_summary_df
display(greedy_results_df[["sample_id", "ground_truth", "predicted", "fen_exact_match", "square_by_square_accuracy"]].head())
display(all_results)


Evaluating Zero-Shot Greedy:   0%|          | 0/40 [00:00<?, ?it/s][transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
Evaluating Zero-Shot Greedy: 100%|██████████| 40/40 [03:53<00:00,  5.83s/it]


Evaluation completed for Zero-Shot Greedy! Results saved to task1_zero-shot_greedy_results.csv.


,sample_id,ground_truth,predicted,fen_exact_match,square_by_square_accuracy
0,sample_000000,6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - -...,e2r1qppp1ppppppppppppppppppppppppppppppppppppp...,0.0,0.0
1,sample_000001,r1r5/p6n/4pq2/3P1p1k/7P/5PQ1/P5P1/R3R1K1 w - -...,b1r1p1pppppppppppppppppppppppppppppppppppppppp...,0.0,0.0
2,sample_000002,8/6pk/5np1/1p5p/p1pP3q/P1P1Q3/1P3P2/5RK1 w - -...,b1rnb1q1ppp1pppppppppppppppppppppppppppppppppp...,0.0,0.0
3,sample_000003,r4r1k/6p1/p5P1/1p2ppNP/1q2b2Q/4n3/PPPRB3/1K4R1...,a1b1c1d1e1f1g1h1i1j1k1l1m1n1o1p1q1r1s1t1u1v1w1...,0.0,0.0
4,sample_000004,4rrk1/6pp/p1p1q3/3p2Q1/2PP2b1/1PR2N2/P5PP/5RK1...,e2r1q1ppp1pppppppppppppppppppppppppppppppppppp...,0.0,0.0


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Zero-Shot Greedy,0.0,3.222376,0.0,173.45


## Experiment 2: Beam search

`num_beams=4` keeps 4 candidate sequences at every step instead of committing to the single best token; `early_stopping=True` stops once all beams have produced an EOS token rather than always running to `max_new_tokens`.

In [6]:
beam_results_df, beam_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=test_split,
    model_name="Zero-Shot Beam Search (n=4)",
    generate_kwargs={"num_beams": 4, "early_stopping": True},
)

all_results = pd.concat([all_results, beam_summary_df], ignore_index=True)
display(beam_results_df[["sample_id", "ground_truth", "predicted", "fen_exact_match", "square_by_square_accuracy"]].head())
display(all_results)


Evaluating Zero-Shot Beam Search (n=4): 100%|██████████| 40/40 [02:22<00:00,  3.57s/it]


Evaluation completed for Zero-Shot Beam Search (n=4)! Results saved to task1_zero-shot_beam_search_(n=4)_results.csv.


,sample_id,ground_truth,predicted,fen_exact_match,square_by_square_accuracy
0,sample_000000,6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - -...,a1b2c3d4e5f6g7h8i9j0,0.0,0.0
1,sample_000001,r1r5/p6n/4pq2/3P1p1k/7P/5PQ1/P5P1/R3R1K1 w - -...,Let's analyze the board step by step.\n\nWe ha...,0.0,0.0
2,sample_000002,8/6pk/5np1/1p5p/p1pP3q/P1P1Q3/1P3P2/5RK1 w - -...,a1b1b2b3b4b5b6b7b8b9b10b11b12b13b14b15b16b17b1...,0.0,0.0
3,sample_000003,r4r1k/6p1/p5P1/1p2ppNP/1q2b2Q/4n3/PPPRB3/1K4R1...,a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6q7r8s9t0u1v2w3...,0.0,0.0
4,sample_000004,4rrk1/6pp/p1p1q3/3p2Q1/2PP2b1/1PR2N2/P5PP/5RK1...,a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6q7r8s9t0u1v2w3...,0.0,0.0


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Zero-Shot Greedy,0.0,3.222376,0.0,173.450
1,Zero-Shot Beam Search (n=4),0.0,1.738887,0.0,92.225


## Experiment 3: Beam search + `no_repeat_ngram_size`

Beam search alone doesn't reliably break the `ppppppp...` repetition loops seen in the greedy baseline, since every beam tends to converge on the same locally-attractive loop. `no_repeat_ngram_size=3` forbids repeating any 3-token sequence, which directly targets that failure mode -- this checks whether it's the missing piece rather than beam width itself.

In [7]:
beam_norepeat_results_df, beam_norepeat_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=test_split,
    model_name="Zero-Shot Beam Search (n=4) + no-repeat-3gram",
    generate_kwargs={"num_beams": 4, "early_stopping": True, "no_repeat_ngram_size": 3},
)

all_results = pd.concat([all_results, beam_norepeat_summary_df], ignore_index=True)
display(beam_norepeat_results_df[["sample_id", "ground_truth", "predicted", "fen_exact_match", "square_by_square_accuracy"]].head())
display(all_results)


Evaluating Zero-Shot Beam Search (n=4) + no-repeat-3gram: 100%|██████████| 40/40 [02:03<00:00,  3.09s/it]


Evaluation completed for Zero-Shot Beam Search (n=4) + no-repeat-3gram! Results saved to task1_zero-shot_beam_search_(n=4)_+_no-repeat-3gram_results.csv.


,sample_id,ground_truth,predicted,fen_exact_match,square_by_square_accuracy
0,sample_000000,6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - -...,a1b2c3d4e5f6g7h8i9j0,0.0,0.0
1,sample_000001,r1r5/p6n/4pq2/3P1p1k/7P/5PQ1/P5P1/R3R1K1 w - -...,Let's analyze the board step by step.\n\nWe ha...,0.0,0.0
2,sample_000002,8/6pk/5np1/1p5p/p1pP3q/P1P1Q3/1P3P2/5RK1 w - -...,a1b2b3b4b5b6b7b8b9b10b11b12b13b14b15b16b17b18b...,0.0,0.0
3,sample_000003,r4r1k/6p1/p5P1/1p2ppNP/1q2b2Q/4n3/PPPRB3/1K4R1...,a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6q7r8s9t0u1v2w3...,0.0,0.0
4,sample_000004,4rrk1/6pp/p1p1q3/3p2Q1/2PP2b1/1PR2N2/P5PP/5RK1...,a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6q7r8s9t0u1v2w3...,0.0,0.0


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Zero-Shot Greedy,0.0,3.222376,0.0,173.450
1,Zero-Shot Beam Search (n=4),0.0,1.738887,0.0,92.225
2,Zero-Shot Beam Search (n=4) + no-repeat-3gram,0.0,1.461025,0.0,76.650


## Comparison

`fen_exact_match` is expected to stay near 0 for a zero-shot 0.8B model regardless of decoding strategy (it requires every character of the FEN to match). `square_by_square_accuracy` and `character_error_rate` are the more informative columns here -- do they move at all between greedy and beam search on this subsample?

In [8]:
sorted_results = all_results.sort_values(by="character_error_rate", ascending=True).reset_index(drop=True)
print(f"--- Decoding Strategy Comparison (Zero-Shot, n={len(test_split)} samples) ---")
display(sorted_results)


--- Decoding Strategy Comparison (Zero-Shot, n=40 samples) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Zero-Shot Beam Search (n=4) + no-repeat-3gram,0.0,1.461025,0.0,76.650
1,Zero-Shot Beam Search (n=4),0.0,1.738887,0.0,92.225
2,Zero-Shot Greedy,0.0,3.222376,0.0,173.450
